## Useful Resources and References

### Cloud-Optimized Data Formats
- [PMTiles Specification](https://github.com/protomaps/PMTiles)
- [Cloud Optimized GeoTIFF](https://www.cogeo.org/)
- [FlatGeoBuf Format](https://flatgeobuf.org/)

### Tools
- [GDAL/OGR Documentation](https://gdal.org/)
- [Tippecanoe (PMTiles creator)](https://github.com/mapbox/tippecanoe)
- [MapLibre GL JS](https://maplibre.org/)

### Performance Optimization
- [HTTP Range Requests MDN](https://developer.mozilla.org/en-US/docs/Web/HTTP/Headers/Range)
- [Cloud Native Geospatial](https://cloudnativegeo.org/)
- [Stacks Documentation](https://purl.stanford.edu/)

### Data Sources Used in This Workshop
- **Vector**: [Santa Clara County RoadCenterLine](https://data.sccgov.org/Transportation/RoadCenterLine/amyq-ahrs) (Socrata Open Data)
- **Reference**: [Stanford Digital Repository - World PMTiles](https://purl.stanford.edu/hf224mw4004) (108 GB scale example)
- **Historical Maps**: [David Rumsey Map Collection](https://www.davidrumsey.com/luna/servlet/detail/RUMSEY~8~1~1578~170036:San-Mateo,-Santa-Cruz,-Santa-Clara) (Optional raster overlay)

### Additional Data Sources
- [Stanford Digital Repository](https://purl.stanford.edu/)
- [OpenStreetMap Data](https://www.openstreetmap.org/)
- [USGS Data Repository](https://www.usgs.gov/products/maps/gis-data)

---

**Completed!** Your data is ready for web deployment. 
Next: Update `index.html` with your GitHub URLs and enable GitHub Pages to go live.

In [ ]:
# Generate deployment information and metadata

# Replace these with your actual GitHub repository information
GITHUB_USERNAME = "YOUR_USERNAME"
GITHUB_REPO = "geo4lib_cloud_optimized"
GITHUB_BRANCH = "main"

# Construct GitHub raw content URLs
pmtiles_url = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/{GITHUB_REPO}/{GITHUB_BRANCH}/data/santa_clara_roads.pmtiles"
github_pages_url = f"https://{GITHUB_USERNAME.lower()}.github.io/{GITHUB_REPO}/"

print("=" * 70)
print("DEPLOYMENT INFORMATION")
print("=" * 70)

print(f"\n1. GITHUB CONFIGURATION:")
print(f"   Username: {GITHUB_USERNAME}")
print(f"   Repository: {GITHUB_REPO}")
print(f"   Branch: {GITHUB_BRANCH}")

print(f"\n2. PMTiles URL (for index.html):")
print(f"   {pmtiles_url}")

print(f"\n3. GitHub Pages URL:")
print(f"   {github_pages_url}")

print(f"\n4. NEXT STEPS:")
print(f"   a) Update GITHUB_USERNAME in this cell to your actual username")
print(f"   b) Push all files to GitHub")
print(f"   c) Enable GitHub Pages (Settings → Pages → main branch)")
print(f"   d) Update placeholder in index.html with the PMTiles URL")
print(f"   e) Browse to the GitHub Pages URL to view your map")

print(f"\n5. VERIFICATION:")
print(f"   • Check browser DevTools Network tab")
print(f"   • Look for 206 Partial Content responses")
print(f"   • These indicate successful HTTP range requests")

print(f"\n6. FILES CREATED:")
print(f"   • index.html - Main web map application")
print(f"   • data/santa_clara_roads.pmtiles - Vector tile dataset")
print(f"   • data/pmtiles-roads.json - Style definition")
print(f"   • README.md - Workshop documentation")
print(f"   • data-preparation.ipynb - This notebook")

# Create a deployment checklist
checklist = {
    "Data Conversion": {
        "✓ Shapefile loaded": True,
        "✓ Transformed to Web Mercator": True,
        "✓ Converted to PMTiles": pmtiles_path.exists() if 'pmtiles_path' in locals() else False,
        "□ Uploaded to GitHub": False,
        "□ Raw URL verified": False,
    },
    "Web Application": {
        "✓ index.html created": True,
        "□ Placeholder URL updated": False,
        "□ Tested locally": False,
        "□ GitHub Pages enabled": False,
        "□ Map displays in browser": False,
    },
    "Performance": {
        "□ Checked Network tab": False,
        "□ Confirmed 206 responses": False,
        "□ Measured load time": False,
    }
}

print(f"\n7. DEPLOYMENT CHECKLIST:")
for category, items in checklist.items():
    print(f"\n   {category}:")
    for task, done in items.items():
        status = "✓" if done else task[0]
        print(f"      {task}")

### Deployment to GitHub Pages

Follow these steps to make your map live on the web:

#### Step 1: Push to GitHub
```bash
git add .
git commit -m "Add cloud-optimized data and map"
git push origin main
```

#### Step 2: Enable GitHub Pages
1. Go to your repository on GitHub
2. Settings → Pages
3. Select "main branch" as source
4. Your site will be published at: `https://[username].github.io/[repo-name]/`

#### Step 3: Update index.html
Replace `PLACEHOLDER_PMTILES_URL` in `index.html` with:
```
https://raw.githubusercontent.com/[username]/[repo-name]/main/data/santa_clara_roads.pmtiles
```

#### Step 4: Browse to Your Map
Open: `https://[username].github.io/[repo-name]/index.html`

### Understanding HTTP Range Requests

When your map loads:
1. Browser requests bytes 0-4096 (PMTiles directory)
2. Server responds with those 4KB (HTTP 206 Partial Content)
3. Browser reads directory to find needed tiles
4. Browser requests only the bytes containing visible tiles
5. Each request is a tiny fraction of the 500MB+ full file

This makes web mapping practical without downloading gigabytes of data!

In [ ]:
# Verify PMTiles file exists and check its properties

logger.info("Verifying PMTiles output...")

if pmtiles_path.exists():
    file_size = pmtiles_path.stat().st_size
    size_mb = file_size / (1024 * 1024)
    size_gb = file_size / (1024 * 1024 * 1024)
    
    print(f"✓ PMTiles file created successfully!")
    print(f"\n  Path: {pmtiles_path}")
    print(f"  File size: {size_mb:.2f} MB" if size_mb < 1024 else f"  File size: {size_gb:.2f} GB")
    print(f"\n  Metadata:")
    print(f"  • Layer name: roads")
    print(f"  • Source: {shapefile_path.name}")
    print(f"  • Features: {len(gdf_web):,}")
    print(f"  • Geometry type: LineString (roads)")
    print(f"  • Coordinate system: Web Mercator (EPSG:3857)")
    print(f"  • Bounds: {gdf_web.total_bounds}")
    
    # Get more details using ogrinfo if available
    print(f"\nDetailed layer information:")
    cmd = ['ogrinfo', '-json', str(pmtiles_path)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and result.stdout:
        try:
            info = json.loads(result.stdout)
            if 'layers' in info and len(info['layers']) > 0:
                layer = info['layers'][0]
                print(f"  • Feature count: {layer.get('featureCount', 'unknown')}")
        except:
            pass
else:
    print("⚠️  PMTiles file not found at expected path:")
    print(f"   {pmtiles_path}")
    print("\nMake sure tippecanoe is installed and the previous cell ran successfully.")

## Section 6: Verify Output and Generate Metadata

### What We'll Do

1. **Verify PMTiles file**: Ensure it was created correctly
2. **Generate metadata**: Document the layer structure
3. **Create deployment instructions**: Show how to host on GitHub Pages
4. **Generate HTTP URLs**: Ready for use in index.html

In [ ]:
# Optional: Create Cloud-Optimized GeoTIFF from raster data
# (Run only if you have raster data to convert)

def create_cog(input_tif_path: Path, output_cog_path: Path) -> bool:
    """
    Convert a GeoTIFF to Cloud-Optimized GeoTIFF format
    
    Parameters:
    - input_tif_path: Path to input GeoTIFF
    - output_cog_path: Path for output COG
    
    Returns:
    - True if successful, False otherwise
    """
    cmd = [
        'gdal_translate',
        '-co', 'DRIVER=GTiff',
        '-co', 'COPY_SRC_OVERVIEWS=YES',  # Copy existing overviews
        '-co', 'TILED=YES',               # Enable internal tiling
        '-co', 'COMPRESS=deflate',        # Compression method
        '-co', 'BIGTIFF=IF_SAFER',        # Support large files
        str(input_tif_path),
        str(output_cog_path)
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0

# Example usage (uncomment if you have a raster file):
# input_raster = DATA_DIR / 'your_raster.tif'
# output_cog = DATA_DIR / 'your_raster_cog.tif'
# 
# if input_raster.exists():
#     print(f"Converting {input_raster} to COG...")
#     if create_cog(input_raster, output_cog):
#         print(f"✓ COG created: {output_cog}")
#     else:
#         print("Error creating COG")
# else:
#     print(f"Raster file not found: {input_raster}")

print("COG creation function defined. Edit the cell below to use it with your raster data.")

## Section 5: Optional - Convert Raster Data to COG

### Cloud-Optimized GeoTIFF (COG)

If you have raster data (satellite imagery, elevation models, historical maps),
convert to COG for efficient remote access.

**Key features:**
- Internal tiling for faster overview rendering
- Overviews (pyramids) for coarser zoom levels
- Optimized byte order (Little Endian on most systems)
- Supported by all GIS software and web mapping libraries

In [ ]:
# Step 2: Create PMTiles from FlatGeoBuf
# We'll use the command-line approach as it's most robust

logger.info("Creating PMTiles from FlatGeoBuf...")

pmtiles_path = DATA_DIR / 'santa_clara_roads.pmtiles'

# Command to create PMTiles using tippecanoe
# Tippecanoe is the standard tool for PMTiles creation
# If not installed: npm install -g tippecanoe

cmd_pmtiles = [
    'tippecanoe',
    '-o', str(pmtiles_path),      # Output PMTiles file
    '-z', '14',                    # Max zoom level (detail level)
    '-Z', '10',                    # Min zoom level (overview level)
    '-l', 'roads',                 # Layer name
    '--drop-densest-as-needed',    # Optimize for size
    str(flatgeobuf_path)           # Input FlatGeoBuf file
]

print(f"Command: {' '.join(cmd_pmtiles)}")
print("\n⚠️  If tippecanoe is not installed, run:")
print("     npm install -g tippecanoe")
print("\n   Then re-run this cell.")

# Try to run tippecanoe
result = subprocess.run(cmd_pmtiles, capture_output=True, text=True)

if result.returncode == 0:
    print(f"\n✓ PMTiles created successfully: {pmtiles_path}")
    print(f"  File size: {pmtiles_path.stat().st_size / (1024*1024):.2f} MB")
    print(f"  Layer: roads")
    print(f"  Zoom levels: 10-14")
else:
    if 'not found' in result.stderr or 'No such file' in result.stderr:
        print("⚠️  tippecanoe not found. Install with:")
        print("     macOS: brew install tippecanoe")
        print("     Ubuntu: apt-get install tippecanoe")
        print("     Or globally via npm: npm install -g tippecanoe")
    else:
        print("Error output:", result.stderr)

In [ ]:
# Step 1: Export shapefile to FlatGeoBuf format
# FlatGeoBuf is an efficient binary format optimized for vector tiles

logger.info("Converting shapefile to FlatGeoBuf...")

# Save our web-mercator data to shapefile first (will use that as source)
temp_shp = SCRATCH_DIR / 'temp_roads_3857.shp'
gdf_web.to_file(temp_shp)

# Path for FlatGeoBuf intermediate format
flatgeobuf_path = SCRATCH_DIR / 'roads.fgb'

# Use GDAL's ogr2ogr to convert
cmd_fgb = [
    'ogr2ogr',
    '-f', 'FlatGeoBuf',  # Output format
    str(flatgeobuf_path),  # Output file
    str(temp_shp),       # Input file
    '-nln', 'roads'      # Layer name
]

print(f"Running: {' '.join(cmd_fgb)}")
result = subprocess.run(cmd_fgb, capture_output=True, text=True)

if result.returncode == 0:
    print(f"✓ FlatGeoBuf created: {flatgeobuf_path}")
    print(f"  File size: {flatgeobuf_path.stat().st_size / 1024:.2f} KB")
else:
    print("Error:", result.stderr)
    print("\nNote: If ogr2ogr is not found, install GDAL:")
    print("  macOS: brew install gdal")
    print("  Ubuntu: sudo apt-get install gdal-bin")

## Section 4: Convert Vector Data to PMTiles

### The Conversion Pipeline

```
Shapefile → FlatGeoBuf → PMTiles
  ↓          ↓            ↓
OGR Format  Streaming    Cloud-ready
           Vector       Tiles with
           Format       Index
```

### Process Steps

1. **Export to FlatGeoBuf**: Convert shapefile to an intermediate streaming format
   - FlatGeoBuf is efficient and supports large datasets
   - Acts as a bridge format that PMTiles tools can read

2. **Create PMTiles**: Generate the final cloud-optimized format
   - PMTiles bundles all tiles in a single file
   - Includes an internal directory at the start of the file
   - Enables HTTP range requests to fetch only needed tiles


In [ ]:
# Validate the data

logger.info("Validating data integrity...")

# Check for null geometries
null_geoms = gdf.geometry.isnull().sum()
print(f"\nData Validation:")
print(f"  • Null geometries: {null_geoms}")

# Check for invalid geometries
invalid_geoms = (~gdf.geometry.is_valid).sum()
print(f"  • Invalid geometries: {invalid_geoms}")

# Check for empty geometries
empty_geoms = (gdf.geometry.is_empty).sum()
print(f"  • Empty geometries: {empty_geoms}")

# Display current CRS
print(f"\nCurrent Coordinate Reference System:")
print(f"  • EPSG Code: {gdf.crs.to_epsg()}")
print(f"  • CRS Name: {gdf.crs.to_string()}")

# Check if already in Web Mercator
if gdf.crs.to_epsg() == 3857:
    print("\n✓ Data is already in Web Mercator (EPSG:3857)")
    gdf_web = gdf.copy()
else:
    print(f"\n→ Transforming from EPSG:{gdf.crs.to_epsg()} to Web Mercator (EPSG:3857)...")
    gdf_web = gdf.to_crs(epsg=3857)
    print("✓ Transformation complete")
    print(f"  New extent: {gdf_web.total_bounds}")

## Section 3: Data Validation and Coordinate System Transformation

### Why Transform Coordinates?

Web maps require data in Web Mercator (EPSG:3857) or similar web-friendly projections.
Our data likely uses a local projected coordinate system (like UTM or State Plane).
We need to transform it to a web-compatible system.

**Key points:**
- Original CRS: Likely UTM or local projection
- Target CRS: EPSG:3857 (Web Mercator, used by most web mapping libraries)
- This transformation preserves accuracy while enabling web compatibility

In [ ]:
# Load the shapefile using GeoPandas
# GeoPandas wraps GDAL/OGR for easier Python data manipulation

logger.info("Loading shapefile into GeoPandas...")
gdf = gpd.read_file(shapefile_path)

print(f"\nDataset Summary:")
print(f"  • Feature count: {len(gdf)}")
print(f"  • Geometry type: {gdf.geometry.type.unique()}")
print(f"  • Coordinate Reference System (CRS): {gdf.crs}")
print(f"  • Spatial extent (bounds):")
bounds = gdf.total_bounds
print(f"    - West: {bounds[0]:.6f}, South: {bounds[1]:.6f}")
print(f"    - East: {bounds[2]:.6f}, North: {bounds[3]:.6f}")

print(f"\nColumn names and types:")
print(gdf.dtypes)

print(f"\nFirst few features:")
print(gdf.head())

In [ ]:
# Examine the shapefile using GDAL command-line tools
# This step inspects the source data without loading it into memory

shapefile_path = DATA_DIR / 'RoadCenterLine_20260610' / 'geo_export_71410037-f1cd-48fb-a6ed-f63e86f313c2.shp'

print(f"Examining shapefile: {shapefile_path}")
print("=" * 70)

# Use ogrinfo to get detailed information
result = subprocess.run(
    ['ogrinfo', str(shapefile_path)],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
else:
    print("Error running ogrinfo. Make sure GDAL is installed:")
    print(result.stderr)
    print("\nInstallation instructions:")
    print("  macOS: brew install gdal")
    print("  Ubuntu: sudo apt-get install gdal-bin")

## Section 2: Examine and Load Source Data

### Step 1: Inspect Shapefile with GDAL

First, we'll use GDAL command-line tools to examine the shapefile structure. This shows us:
- Layer names and feature counts
- Geometry types
- Coordinate reference system (CRS)
- Attribute fields and their data types
- Spatial extent (bounding box)


In [ ]:
# Import required libraries

import os
import subprocess
import json
from pathlib import Path
from typing import Dict, Tuple

# Geospatial libraries
import geopandas as gpd
import rasterio
from rasterio.io import MemoryFile
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

# Data manipulation
import pandas as pd
from shapely.geometry import box

# System utilities
import logging
import warnings
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Define project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
SCRATCH_DIR = PROJECT_ROOT / 'scratch'

# Ensure directories exist
DATA_DIR.mkdir(exist_ok=True)
SCRATCH_DIR.mkdir(exist_ok=True)

print("✓ All libraries imported successfully!")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

In [ ]:
# Install required packages
# Run this cell first to set up your environment

import subprocess
import sys

packages = [
    'geopandas',      # Vector data manipulation
    'rasterio',       # Raster data handling
    'shapely',        # Geometric operations
    'pandas',         # Data analysis
    'pyproj',         # Coordinate reference systems
]

print("Installing required packages...")
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✓ Installation complete!")
print("\nNote: GDAL needs to be installed separately:")
print("  - macOS: brew install gdal")
print("  - Ubuntu: sudo apt-get install gdal-bin")
print("  - Windows: Download from https://trac.osgeo.org/gdal/wiki/DownloadingGdalBinaries")

## Section 1: Install Dependencies and Import Libraries

# Cloud-Optimized Data Preparation Workshop

## Overview
This notebook walks through converting traditional geospatial data formats into cloud-optimized alternatives that enable efficient web-based visualization via HTTP range requests.

### Data Sources Used in This Workshop

This workshop uses publicly available data:

1. **Santa Clara County Road Network (Vector)** - [RoadCenterLine Dataset](https://data.sccgov.org/Transportation/RoadCenterLine/amyq-ahrs)
   - Public transportation infrastructure data
   - Open Data portal (Socrata)
   - Available in multiple formats including shapefile

2. **Stanford Digital Repository Reference** - [World PMTiles](https://purl.stanford.edu/hf224mw4004)
   - Large-scale example of cloud-optimized data (108 GiB)
   - Demonstrates the full potential of PMTiles format
   - Live URL: `https://stacks.stanford.edu/file/druid:hf224mw4004/20231116.pmtiles`

3. **David Rumsey Historical Maps (Optional Raster)** - [Santa Clara County Maps](https://www.davidrumsey.com/luna/servlet/detail/RUMSEY~8~1~1578~170036:San-Mateo,-Santa-Cruz,-Santa-Clara)
   - Georeferenced historical maps for comparison with contemporary data
   - Example of raster data suitable for COG conversion
   - Demonstrates temporal analysis capabilities

### What We'll Do
1. **Examine source data** using GDAL command-line tools
2. **Convert vector data** (shapefiles) → FlatGeoBuf → PMTiles
3. **Optimize raster data** (GeoTIFF) → Cloud-Optimized GeoTIFF (COG)
4. **Generate metadata** and prepare URLs for web deployment

### Key Formats
- **PMTiles**: Vector tile container with embedded index (for efficient HTTP range requests)
- **COG**: Cloud-Optimized GeoTIFF (raster data with optimized tiling)
- **FlatGeoBuf**: Intermediate vector format supporting streaming

### Output
- `santa_clara_roads.pmtiles` - Vector tile dataset of road network
- Styled visualization via `pmtiles-roads.json`
- HTTP URLs ready for web map deployment